# 07 — Survival Analysis (WIP)

**Status: Work in Progress**

The CLV model in notebook 06 uses the simple formula:

> CLV = (Monthly Revenue × Gross Margin) / Monthly Churn Rate

This works well as a quick estimate, but it has a key assumption: **the churn rate is constant over time**.

In reality, churn risk is not constant — it's highest in the first few months and decreases for customers who stick around. Survival analysis lets us model this time-varying risk properly.

This notebook will implement:
1. **Kaplan-Meier survival curves** — non-parametric estimate of survival by segment
2. **Cox Proportional Hazards model** — regression model for churn hazard rate
3. **Survival-based CLV** — more accurate CLV using the survival function

**TODO:**
- [ ] Install `lifelines` library
- [ ] Format data for survival analysis (duration = tenure, event = churn)
- [ ] Fit Kaplan-Meier by contract type
- [ ] Fit Cox model with covariates
- [ ] Compare survival-based CLV vs simple CLV from notebook 06


In [ ]:
# pip install lifelines
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import preprocess

df = preprocess('../data/raw/telco_churn.csv')
print(df.shape)

## Survival Analysis Setup

In survival analysis terms:
- **Duration** = `tenure` (how long the customer has been a subscriber)
- **Event** = `Churn` = 1 (the event we're tracking is cancellation)
- Customers with `Churn` = 0 are **right-censored** — we don't know when/if they'll churn, only that they haven't yet

In [ ]:
# Prepare survival data
survival_df = df[['tenure', 'Churn', 'Contract', 'InternetService',
                  'MonthlyCharges', 'senior_citizen']].copy()

# Duration must be at least 1 for lifelines
survival_df['duration'] = survival_df['tenure'].clip(lower=1)
survival_df['event'] = survival_df['Churn']  # 1 = churned, 0 = censored

survival_df.head()

## Kaplan-Meier Survival Curves

The Kaplan-Meier estimator gives us a non-parametric estimate of the **survival function** S(t) — the probability that a customer is still active at time t.

We'll plot separate curves by contract type to visually confirm what we saw in the cohort analysis.

In [ ]:
# TODO: this cell requires lifelines
# from lifelines import KaplanMeierFitter
#
# fig, ax = plt.subplots(figsize=(10, 6))
# colors = {'Month-to-month': '#e74c3c', 'One year': '#f39c12', 'Two year': '#2ecc71'}
#
# for contract_type, group in survival_df.groupby('Contract'):
#     kmf = KaplanMeierFitter()
#     kmf.fit(
#         group['duration'],
#         event_observed=group['event'],
#         label=contract_type
#     )
#     kmf.plot_survival_function(ax=ax, color=colors.get(contract_type, 'gray'), ci_show=True)
#
# ax.set_title('Kaplan-Meier Survival Curves by Contract Type')
# ax.set_xlabel('Months Since Signup')
# ax.set_ylabel('Survival Probability (% Still Active)')
# ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
# plt.tight_layout()
# plt.show()

print('Uncomment after installing lifelines: pip install lifelines')

## Cox Proportional Hazards Model

The Cox model estimates the **hazard rate** — the instantaneous risk of churn at time t — as a function of customer covariates.

Unlike logistic regression which predicts a binary churn/no-churn outcome, Cox regression models **when** a customer will churn, which gives us richer insight.

The hazard ratio for each variable tells us how much that variable multiplies the churn risk:
- HR > 1 → increases churn risk
- HR < 1 → decreases churn risk (protective)
- HR = 1 → no effect

In [ ]:
# TODO: Cox model — requires lifelines
# from lifelines import CoxPHFitter
#
# cox_df = pd.get_dummies(survival_df, columns=['Contract', 'InternetService'], drop_first=True)
# cox_df = cox_df[['duration', 'event', 'MonthlyCharges', 'senior_citizen',
#                   'Contract_One year', 'Contract_Two year',
#                   'InternetService_Fiber optic', 'InternetService_No']]
#
# cph = CoxPHFitter()
# cph.fit(cox_df, duration_col='duration', event_col='event')
# cph.print_summary()

print('TODO: fit Cox model')

## Survival-Based CLV (TODO)

Once we have the survival function S(t), CLV can be calculated as:

$$CLV = \sum_{t=1}^{T} \frac{R \cdot m \cdot S(t)}{(1+d)^t}$$

Where:
- $R$ = monthly revenue
- $m$ = gross margin
- $S(t)$ = probability of still being active at month t (from KM curve)
- $d$ = monthly discount rate (e.g. 10% annual = 0.83% monthly)
- $T$ = max time horizon (e.g. 60 months)

This gives a more accurate CLV by accounting for the fact that churn risk is **not constant** over the customer lifecycle.

**TODO:** implement and compare to simple CLV from notebook 06.